# Grid overzicht en 3-fasige fout op K2

Deze notebook gebruikt het net uit `main.py` en toont alleen het overzicht en het resultaat van een 3-fasige fout op `K2`.

In [11]:
import importlib
import warnings

import pandapower as pp
import pandapower.shortcircuit as sc

warnings.filterwarnings("ignore", category=FutureWarning, module="pandapower")

import main
importlib.reload(main)
create_grid = main.create_grid

## 1. Net maken

In [12]:
net = create_grid()
net

This pandapower network includes the following parameter tables:
   - bus (3 elements)
   - load (1 element)
   - sgen (1 element)
   - gen (1 element)
   - ext_grid (1 element)
   - line (1 element)
   - trafo (1 element)

## 2. Overzicht

In [13]:
print("Bussen")
display(net.bus[["name", "vn_kv", "in_service"]])

print("\nExtern net")
display(net.ext_grid[["name", "bus", "vm_pu", "s_sc_max_mva", "rx_max"]])

print("\nTransformatoren")
display(net.trafo[["name", "hv_bus", "lv_bus", "sn_mva", "vn_hv_kv", "vn_lv_kv", "vk_percent", "vkr_percent"]])

print("\nLijnen")
display(net.line[["name", "from_bus", "to_bus", "length_km", "r_ohm_per_km", "x_ohm_per_km", "c_nf_per_km", "max_i_ka"]])

print("\nBelastingen")
display(net.load[["name", "bus", "p_mw", "q_mvar"]])

print("\nAsynchrone motoren")
display(net.sgen.reindex(columns=["name", "bus", "p_mw", "q_mvar", "sn_mva", "type", "k", "rx"]))

print("\nGeneratoren")
display(net.gen[["name", "bus", "p_mw", "vm_pu", "sn_mva", "vn_kv", "xdss_pu", "rdss_ohm", "cos_phi"]])

Bussen


,name,vn_kv,in_service
0,K1,150.0,True
1,K2,10.0,True
2,K3,10.0,True



Extern net


,name,bus,vm_pu,s_sc_max_mva,rx_max
0,150 kV Grid,0,1.0,10392.0,0.0



Transformatoren


...

,name,bus,p_mw,q_mvar
0,Load,2,2.0,1.5



Asynchrone motoren


,name,bus,p_mw,q_mvar,sn_mva,type,k,rx
0,Motor,2,-2.105263,-1.304725,2.47678,motor,5.0,0.1



Generatoren


,name,bus,p_mw,vm_pu,sn_mva,vn_kv,xdss_pu,rdss_ohm,cos_phi
0,Generator,2,1.7,1.0,2.0,10.0,0.2,0.35,0.85


## 3. Equivalente impedanties


In [15]:
import math
import pandas as pd

def z_from_pq(vn_kv, p_mw, q_mvar):
    return vn_kv**2 / complex(p_mw, -q_mvar)

rows = []

for _, load in net.load.iterrows():
    bus_vn = net.bus.at[load.bus, "vn_kv"]
    z = z_from_pq(bus_vn, load.p_mw, load.q_mvar)
    rows.append({
        "component": load["name"],
        "model": "belasting bedrijfs-equivalent",
        "bus": net.bus.at[load.bus, "name"],
        "r_ohm": z.real,
        "x_ohm": z.imag,
        "z_abs_ohm": abs(z),
    })

for _, motor in net.sgen[net.sgen["type"].eq("motor")].iterrows():
    bus_vn = net.bus.at[motor.bus, "vn_kv"]
    p_load = -motor.p_mw
    q_load = -motor.q_mvar
    z_bedrijf = z_from_pq(bus_vn, p_load, q_load)
    z_start_abs = (bus_vn**2 / motor.sn_mva) / motor.k
    x_start = z_start_abs / math.sqrt(1 + motor.rx**2)
    r_start = motor.rx * x_start
    rows.extend([
        {
            "component": motor["name"],
            "model": "motor bedrijfs-equivalent",
            "bus": net.bus.at[motor.bus, "name"],
            "r_ohm": z_bedrijf.real,
            "x_ohm": z_bedrijf.imag,
            "z_abs_ohm": abs(z_bedrijf),
        },
        {
            "component": motor["name"],
            "model": "motor aanloop/kortsluit-equivalent",
            "bus": net.bus.at[motor.bus, "name"],
            "r_ohm": r_start,
            "x_ohm": x_start,
            "z_abs_ohm": z_start_abs,
        },
    ])

for _, gen in net.gen.iterrows():
    z_base = gen.vn_kv**2 / gen.sn_mva
    r_gen = gen.rdss_ohm
    x_gen = gen.xdss_pu * z_base
    rows.append({
        "component": gen["name"],
        "model": "generator subtransient-equivalent",
        "bus": net.bus.at[gen.bus, "name"],
        "r_ohm": r_gen,
        "x_ohm": x_gen,
        "z_abs_ohm": abs(complex(r_gen, x_gen)),
    })

z_eq = pd.DataFrame(rows)
display(z_eq.round(6))

,component,model,bus,r_ohm,x_ohm,z_abs_ohm
0,Load,belasting bedrijfs-equivalent,K3,32.000000,24.000000,40.000000
1,Motor,motor bedrijfs-equivalent,K3,34.318750,21.268851,40.375000
2,Motor,motor aanloop/kortsluit-equivalent,K3,0.803493,8.034925,8.075000
3,Generator,generator subtransient-equivalent,K3,0.350000,10.000000,10.006123


## 4. 3-fasige fout op K3

In [ ]:
k3_bus = net.bus.index[net.bus["name"].eq("K3")][0]

sc.calc_sc(
    net,
    bus=[k3_bus],
    fault="3ph",
    case="max",
)

net.res_bus_sc.loc[[k3_bus]]